In [21]:
import torch
from torchvision import transforms as T
from torch.utils.data import DataLoader, Dataset, Subset
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchmetrics

import pytorch_lightning as L
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd
from PIL import Image

In [22]:
class ImageDataset(Dataset):
    def __init__(self, csv_file, transform=None, is_test=False):
        self.df = pd.read_csv(csv_file)
        self.transform = transform
        self.is_test = is_test

        if not self.is_test:
            self.classes = [0,1]

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, img_path
        else:
            label = row['label']
            return image, torch.tensor(label, dtype=torch.long)

In [23]:
class ImageDataModule(L.LightningDataModule):
    def __init__(
        self, train_csv, test_csv, batch_size=32, img_size=224, val_split=0.2
    ):
        super().__init__()
        self.train_csv = train_csv
        self.test_csv = test_csv
        self.batch_size = batch_size
        self.img_size = img_size
        self.val_split = val_split

        self.transform = T.Compose([
            T.Resize((img_size,img_size)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
            )
        ])

    def setup(self, stage=None):
        full_dataset = ImageDataset(self.train_csv, transform=self.transform)
        self.num_classes = 2

        train_idx, val_idx = train_test_split(
            range(len(full_dataset)),
            test_size=self.val_split,
            random_state=42,
            stratify=full_dataset.df['label']
        )

        self.train_dataset = Subset(full_dataset, train_idx)

        val_dataset_eval = ImageDataset(self.train_csv, transform=self.transform)
        self.val_dataset = Subset(val_dataset_eval, val_idx)

        self.test_dataset = ImageDataset(self.test_csv, transform=self.transform, is_test=True)

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=4,
            persistent_workers=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=4,
            persistent_workers=True
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=4,
            persistent_workers=True
        )

In [24]:
class ImageClassifier(L.LightningModule):
    def __init__(self, lr=1e-3, threshold=0.5):
        super().__init__()
        self.save_hyperparameters()

        self.backbone = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, 1)

        self.criterion = nn.BCEWithLogitsLoss()

        self.train_acc = torchmetrics.Accuracy(task='binary')
        self.val_acc = torchmetrics.Accuracy(task='binary')

    def forward(self, x):
        return self.backbone(x)

    def _step(self, bacth, stage):
            x, y = bacth
            logits = self(x)
            targets = y.float().unsqueeze(1)
            loss = self.criterion(logits, targets)

            probs = torch.sigmoid(logits).squeeze(1)
            y_int = y.int()
            self.log(f'{stage}_loss', loss, prog_bar=True)
            return loss
    
    def training_step(self, batch):
        return self._step(batch, 'train')

    def validation_step(self, batch):
        return self._step(batch, 'val')

    def predict_step(self, batch):
        imgs, paths = batch
        probs = torch.sigmoid(self(imgs)).squeeze(1)
        preds = (probs >= self.hparams.threshold).long()
        return paths, preds
        


    def configure_optimizers(self):
        opt = optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=1e-4)
        sch = optim.lr_scheduler.ReduceLROnPlateau(optimizer=opt, mode='min', factor=0.1, patience=2)

        return {
            'optimizer':opt,
            'lr_scheduler' :{
                'scheduler': sch,
                'monitor': 'val_loss'
            }
        }    

In [25]:

dm = ImageDataModule(
    train_csv='train.csv', test_csv='test.csv', batch_size=32, img_size=224
)
dm.setup()

model = ImageClassifier(lr=3e-4)

checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    dirpath='checkpoints/',
    filename='arbin-resnet-{epoch:02d}-{val_loss:.4f}',
    save_top_k=1,
    mode='min'
)

early_stop_callback = EarlyStopping(
    monitor='val_loss', patience=5, mode='min'
)

trainer = L.Trainer(
    max_epochs=20,
    accelerator='auto',
    devices=1,
    callbacks=[checkpoint_callback, early_stop_callback],
    precision='16-mixed'
)

trainer.fit(model, dm)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/moldo/Projects/MLCompete/Medium/ECG/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name      | Type              | Params | Mode  | FLOPs
----------------------------------------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 8: 100%|██████████| 100/100 [00:09<00:00, 10.94it/s, v_num=6, train_loss=0.00032, val_loss=0.0137]  


In [26]:
best_model_path = checkpoint_callback.best_model_path
model = ImageClassifier.load_from_checkpoint(best_model_path)

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

test_loader = dm.test_dataloader()
results = []

with torch.no_grad():
    for images, paths in test_loader:
        images = images.to(device)
        probs = torch.sigmoid(model(images)).squeeze(1)
        preds = (probs >= model.hparams.threshold).long().cpu().numpy()

        for img_path, pred in zip(paths, preds):
            results.append({
                'image_path': img_path,
                'label': int(pred)
            })


submission_df = pd.DataFrame(results)
submission_df.to_csv('submission.csv', index=False)